<a href="https://colab.research.google.com/github/Davidson-Open-University/csc203-labs/blob/main/notebooks/CSC203-Lab4-Partial-Orders-Lattices-and-Boolean-Algebra.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 203 Lab 4: Partial Orders, Lattices and Boolean Algebra

**Study Session 4: Boolean Algebra and Lattices** · Davidson Open University Global

**In this lab you will:**
- test whether a relation is reflexive, antisymmetric and transitive;
- draw the Hasse diagram of a partial order;
- find joins (least upper bounds) and meets (greatest lower bounds) in a lattice;
- simplify Boolean expressions with SymPy.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the examples and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short Python function and test it.
- **Part D – Apply and reflect** (about 10 minutes): use your tools on your own example and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `csc203-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

*This lab is practice to help you learn and prepare for the assignment. It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), 6) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k) + '=' + _norm(v) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), 6) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('csc203|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False)
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '4.1': {'h': '5b2648c74deece5f', 'hint': 'd divides n when n % d == 0. Include both 1 and n.', 'ok': 'Your divisors function works.', 'args': [(12,), (17,), (36,), (1,)]},
    '4.2': {'h': 'f9c5b96d6c4ae2e4', 'hint': 'Join = lcm, meet = gcd. Check that both answers divide 18.', 'ok': 'Both answers are divisors of 18, so the join and meet stay inside the lattice.'},
    '4.3': {'h': '03d2db982b8422d4', 'hint': 'A prime has exactly two divisors: len(divisors(d)) == 2.', 'ok': 'A finite Boolean algebra is determined by its atoms: it is isomorphic to the power set of its atoms.'},
    '4.4': {'h': '745c936ba328f04b', 'hint': 'Every element must be related to itself.', 'args': [({1, 2, 3}, {(1, 1), (2, 2), (3, 3), (1, 2)}), ({1, 2, 3}, {(1, 1), (2, 2)}), ({1}, {(1, 1)}), ({1, 2}, set())]},
    '4.5': {'h': '35a7dcbd04ab5377', 'hint': 'Look for a pair (a, b) with a ≠ b where the reverse pair (b, a) is also in R. If you find one, the relation is not antisymmetric.', 'args': [({1, 2}, {(1, 2), (2, 1)}), ({1, 2}, {(1, 1), (1, 2)}), ({1, 2, 3}, {(1, 2), (2, 3), (1, 3)}), ({1}, {(1, 1)})]},
    '4.6': {'h': '99d30b89969a9ea4', 'hint': 'Use two loops over R: for (a, b) in R for (b2, c) in R if b == b2, check (a, c) in R.', 'ok': 'Now you can test any relation for being a partial order.', 'args': [({1, 2, 3}, {(1, 2), (2, 3)}), ({1, 2, 3}, {(1, 2), (2, 3), (1, 3)}), ({1, 2}, {(1, 2), (2, 1), (1, 1), (2, 2)}), ({1, 2}, {(1, 2), (2, 1)})]},
    '4.7': {'h': 'f241151889acb33a', 'hint': 'Type the expression with &, | and ~, then wrap it in simplify_logic(...).', 'ok': 'Fewer gates means a cheaper, faster switching circuit.', 'logic': True},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
A relation R on a set S is a **partial order** if it is
- **reflexive:** (a, a) ∈ R for every a ∈ S;
- **antisymmetric:** if (a, b) ∈ R and (b, a) ∈ R, then a = b;
- **transitive:** if (a, b) ∈ R and (b, c) ∈ R, then (a, c) ∈ R.

"a divides b" on the positive divisors of a number is a classic partial order. Run the cell to build it for the divisors of 12 and draw its **Hasse diagram**: the diagram keeps only the "covering" edges and puts larger elements higher up.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

S = [d for d in range(1, 13) if 12 % d == 0]
R = {(a, b) for a in S for b in S if b % a == 0}
print('Divisors of 12:', S)
print('Number of pairs in R:', len(R))

# Hasse diagram: remove loops and edges implied by transitivity
H = nx.transitive_reduction(nx.DiGraph([(a, b) for (a, b) in R if a != b]))

# Height of each number = how many prime factors it has (so 1 is at the bottom)
def height(v):
    count, d = 0, 2
    while v > 1:
        if v % d == 0:
            v, count = v // d, count + 1
        else:
            d += 1
    return count

pos = {}
for h in set(height(v) for v in S):
    row = sorted(v for v in S if height(v) == h)
    for i, v in enumerate(row):
        pos[v] = (i - (len(row) - 1) / 2, h)
nx.draw(H, pos, with_labels=True, arrows=False, node_color='#cfe2ff', node_size=800)
plt.title('Hasse diagram of the divisors of 12')
plt.show()

**Read the diagram.** a ≼ b when you can go **up** from a to b. The **join** a ∨ b is the lowest element above both; the **meet** a ∧ b is the highest element below both. For divisibility, join = lcm (least common multiple) and meet = gcd (greatest common divisor).

In [ ]:
from math import gcd
def lcm(a, b):
    return a * b // gcd(a, b)
print('join of 4 and 6 = lcm =', lcm(4, 6), '   meet of 4 and 6 = gcd =', gcd(4, 6))

## Part B – Practise (about 20 minutes)
**B1.** Complete `divisors(n)` so that it returns the **sorted list** of positive divisors of n. *Hint:* a list comprehension, `[d for d in range(1, n + 1) if ...]`.

In [ ]:
def divisors(n):
    return ...      # YOUR CODE

print(divisors(12))

In [ ]:
check('4.1', divisors)

**B2.** In the lattice of divisors of 18 under divisibility, find the join and the meet of 6 and 9. Give the answer as a tuple `(join, meet)`.

In [ ]:
answer_4_2 = ...   # (join of 6 and 9, meet of 6 and 9)

In [ ]:
check('4.2', answer_4_2)

**B3. Atoms.** An **atom** is an element that sits directly above the least element 1 (it covers 1). In the divisor lattice, the atoms are the **prime** divisors. Make `answer_4_3` the **set** of atoms of the divisors of 60.

In [ ]:
answer_4_3 = ...

In [ ]:
check('4.3', answer_4_3)

## Part C – Build a tool (about 20 minutes)
Complete the three property tests. Each takes a set `S` and a relation `R` (a set of pairs) and returns `True` or `False`. Then the cell tests them on several relations.

*Hints:*
- reflexive: `all((a, a) in R for a in S)`
- antisymmetric: for every (a, b) in R with a ≠ b, the pair (b, a) must **not** be in R
- transitive: for every (a, b) and (b2, c) in R with b == b2, the pair (a, c) must be in R

In [ ]:
def is_reflexive(S, R):
    ...      # YOUR CODE

In [ ]:
check('4.4', is_reflexive)

In [ ]:
def is_antisymmetric(S, R):
    ...      # YOUR CODE

In [ ]:
check('4.5', is_antisymmetric)

In [ ]:
def is_transitive(S, R):
    ...      # YOUR CODE

In [ ]:
check('4.6', is_transitive)

In [ ]:
def is_partial_order(S, R):
    return bool(is_reflexive(S, R)) and bool(is_antisymmetric(S, R)) and bool(is_transitive(S, R))

d12 = divisors(12)
S = set(d12) if isinstance(d12, list) else set()
tests = {
    'divides':             {(a, b) for a in S for b in S if b % a == 0},
    'less than or equal':  {(a, b) for a in S for b in S if a <= b},
    'less than':           {(a, b) for a in S for b in S if a < b},
    'same parity':         {(a, b) for a in S for b in S if a % 2 == b % 2},
}
for name, R in tests.items():
    print(f'{name:20} partial order? {is_partial_order(S, R)}')

### C2. Boolean algebra with SymPy
SymPy can simplify Boolean expressions. It writes ∧ as `&`, ∨ as `|` and complement (′) as `~`.
Use `simplify_logic` to simplify (a ∧ b) ∨ (a ∧ b′) ∨ (a′ ∧ b). Store the **simplified** expression in `answer_4_7`.

In [ ]:
from sympy import symbols
from sympy.logic import simplify_logic
a, b, c = symbols('a b c')
answer_4_7 = ...     # simplify_logic( ... )
print(answer_4_7)

In [ ]:
check('4.7', answer_4_7)

## Part D – Apply and reflect (about 10 minutes)
**D1. Prepare for Assignment 4.** Use your functions to check that divisibility is a partial order on the divisors of **30**, and draw its Hasse diagram by changing 12 to 30 in the Part A cell. Compare it with the diagram you draw by hand.

In [ ]:
# D1: divisors of 30

**D2. Reflect.** Answer in one or two sentences each:
1. Why is "less than" (<) not a partial order, but "less than or equal" (≤) is?
2. The divisors of 30 form a Boolean algebra, but the divisors of 12 do not. What is different? (*Hint:* look at the complements, or compare the number of elements with 2^(number of atoms).)
3. Give one real-life example of a partial order that is not a total order.

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `csc203-labs` repository → commit message *"Lab 4 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.